In [1]:
#Notebook 2
## Data Cleaning and Preprocessing
import pandas as pd
import numpy as np

from pathlib import Path

import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")

##Project Directories
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
FIELD_DIR = DATA_DIR / "field"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Processed directory:", PROCESSED_DIR.resolve())
print("Model directory:", MODEL_DIR.resolve())

Libraries loaded successfully.
Project root: C:\Users\HP\Documents\SP-XGBOOST
Processed directory: C:\Users\HP\Documents\SP-XGBOOST\data\processed
Model directory: C:\Users\HP\Documents\SP-XGBOOST\models


In [2]:
#Load the Clean Dataset
df = pd.read_csv(
    FIELD_DIR / "SPhealth_student_data.csv"
)


print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (569, 61)


In [3]:
#Inspect the Dataset
df.head()
#df.info()

#Remove Student ID
ID_COLUMN = "SID"
#TARGET = "RISK_LABEL"
df = df.drop(columns=[ID_COLUMN])
print(df.columns.tolist())

#Verify Leakage Variables
leakage_columns = [
    "CGPA",
    "GPA_S2",
    "GPA_LAST",
    "FAIL_COUNT",
    "WARN_STATUS",
    "REPEAT_COUNT"
]

remaining_leakage = [
    col for col in leakage_columns
    if col in df.columns
]

print("Leakage variables still present:")
print(remaining_leakage)

['GPA_S1', 'CA_AVG', 'EXAM_AVG', 'CLIN_AVG', 'LAB_AVG', 'ATT_RATE', 'ASSIGN_LATE', 'Age_group', 'Gender', 'Year_study', 'Programme', 'SES', 'Financial_diff', 'Employment_hrs', 'Study_hrs_day', 'Sleep_hrs', 'Self_risk_percep', 'Reviews notes within 24h', 'Understands content pre-exam', 'Seeks help when stuck', 'Uses library regularly', 'Completes readings', 'Takes organised notes', 'Concentration in self-study', 'Participates in class', 'Clinical takes study time', 'Prepared for clinical assess', 'Rotations affect performance', 'Adequate supervision', 'Schedule conflicts', 'Confident in clinical skills', 'Anxious about assessments', 'Sleep difficulty', 'Burnt out', 'Hopeless/unmotivated', 'Physical health affected', 'Considered break', 'Emotionally supported', 'Lecturers approachable', 'Sleep affects concentration', 'Regular exercise', 'Balanced diet', 'Health interferes studies', 'Takes rest breaks', 'Sense of belonging', 'Participates in clubs', 'Adequate advisory support', 'Concerns 

In [4]:
#Verify Semester-1 Predictors
semester1_features = [
    "GPA_S1",
    "CA_AVG",
    "EXAM_AVG",
    "CLIN_AVG",
    "LAB_AVG",
    "ATT_RATE"
]

print("Semester-1 predictors:")
for col in semester1_features:
    if col in df.columns:
        print("✓", col)
    else:
        print("✗", col, "MISSING")

#Check RISK_LABEL
print("=" * 70)
print(df["RISK_LABEL"].unique())

print("\nTarget distribution:")
print(df["RISK_LABEL"].value_counts(dropna=False))

Semester-1 predictors:
✓ GPA_S1
✓ CA_AVG
✓ EXAM_AVG
✓ CLIN_AVG
✓ LAB_AVG
✓ ATT_RATE
['Not At-Risk' 'At-Risk']

Target distribution:
RISK_LABEL
Not At-Risk    412
At-Risk        157
Name: count, dtype: int64


In [5]:
#Data validation
#Check Missing Values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(
    ascending=False
)
print(missing)

#Check Duplicate Rows
print("Number of duplicate rows:", df.duplicated().sum())

Series([], dtype: int64)
Number of duplicate rows: 0


In [6]:
#Check Constant Columns
constant_columns = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1
]
print("Constant columns:")
print(constant_columns)
#Remove
df = df.drop(
    columns=constant_columns
)
print("Shape after removing constant columns:", df.shape)

Constant columns:
['Programme']
Shape after removing constant columns: (569, 59)


In [7]:
#Separate Target from Predictors
X = df.drop(
    columns=["RISK_LABEL"]
).copy()

y = df["RISK_LABEL"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

#Make Sure Target Has No Missing Values
print("Missing target values:", y.isna().sum())

print("Target dtype:", y.dtype)
print("Target values:", y.unique())
#map Target values
y = y.map({
    "Not At-Risk": 0,
    "At-Risk": 1
})

#verify
print(y.value_counts())
print(y.unique())

X shape: (569, 58)
y shape: (569,)
Missing target values: 0
Target dtype: object
Target values: ['Not At-Risk' 'At-Risk']
RISK_LABEL
0    412
1    157
Name: count, dtype: int64
[0 1]


In [8]:
#Train/Test Split --split before fitting the encoder
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
#Check
print("Training:", X_train.shape)
print("Testing :", X_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())


Training: (455, 58)
Testing : (114, 58)

Training target:
RISK_LABEL
0    329
1    126
Name: count, dtype: int64

Testing target:
RISK_LABEL
0    83
1    31
Name: count, dtype: int64


In [9]:
#Check Object Columns in Training Data
object_columns = X_train.select_dtypes(
    include="object"
).columns.tolist()
print("Object columns:")
print(object_columns)

#Verify Their Categories
for col in object_columns:
    print("=" * 70)
    print(col)
    print(sorted(X_train[col].dropna().unique()))


Object columns:
['ASSIGN_LATE', 'Age_group', 'Gender', 'SES', 'Financial_diff', 'Employment_hrs', 'Study_hrs_day', 'Sleep_hrs', 'Self_risk_percep']
ASSIGN_LATE
['Almost never', 'Rarely', 'Sometimes']
Age_group
['18-20 years', '21-23 years', '24-26 years', '27-29 years', '30-35 years']
Gender
['Female', 'Male']
SES
['High income', 'Low income', 'Middle income', 'Upper-middle income', 'Very low income']
Financial_diff
['Always', 'Never', 'Often (weekly)', 'Rarely (once or twice a semester)', 'Sometimes (monthly)']
Employment_hrs
['No', 'Yes - 10-20 hours/week', 'Yes - fewer than 10 hours/week', 'Yes - more than 20 hours/week']
Study_hrs_day
['0', '1-2 hours', '2-3 hours', '3-4 hours', 'Less than 1 hour', 'More than 4 hours']
Sleep_hrs
['4-5 hours', '5-6 hours', '6-7 hours', '7-8 hours', 'Less than 4 hours', 'More than 8 hours']
Self_risk_percep
['High risk', 'Low risk', 'Moderate risk', 'Very low risk']


In [10]:
#Define the Ordinal Mappings
ordinal_mapping = {

    "ASSIGN_LATE": [
        "Almost never",
        "Rarely",
        "Sometimes"
    ],

    "Age_group": [
        "18-20 years",
        "21-23 years",
        "24-26 years",
        "27-29 years",
        "30-35 years"
    ],

    "SES": [
        "Very low income",
        "Low income",
        "Middle income",
        "Upper-middle income",
        "High income"
    ],

    "Financial_diff": [
        "Never",
        "Rarely (once or twice a semester)",
        "Sometimes (monthly)",
        "Often (weekly)",
        "Always"
    ],

    "Employment_hrs": [
        "No",
        "Yes - fewer than 10 hours/week",
        "Yes - 10-20 hours/week",
        "Yes - 20-20 hours/week",
        "Yes - more than 20 hours/week"
    ],

    "Study_hrs_day": [
        "0",
        "Less than 1 hour",
        "1-2 hours",
        "2-3 hours",
        "3-4 hours",
        "More than 4 hours"
    ],

    "Sleep_hrs": [
        "Less than 4 hours",
        "4-5 hours",
        "5-6 hours",
        "6-7 hours",
        "7-8 hours",
        "More than 8 hours"
    ],

    "Self_risk_percep": [
        "Very low risk",
        "Low risk",
        "Moderate risk",
        "High risk"
    ]
}

#Check Mapping Against Actual Data --Before encoding
for col, categories in ordinal_mapping.items():

    print("=" * 70)
    print(col)

    actual = set(X_train[col].dropna().unique())
    expected = set(categories)

    print("Unexpected values:")
    print(actual - expected)

    print("Missing expected categories:")
    print(expected - actual)


ASSIGN_LATE
Unexpected values:
set()
Missing expected categories:
set()
Age_group
Unexpected values:
set()
Missing expected categories:
set()
SES
Unexpected values:
set()
Missing expected categories:
set()
Financial_diff
Unexpected values:
set()
Missing expected categories:
set()
Employment_hrs
Unexpected values:
set()
Missing expected categories:
{'Yes - 20-20 hours/week'}
Study_hrs_day
Unexpected values:
set()
Missing expected categories:
set()
Sleep_hrs
Unexpected values:
set()
Missing expected categories:
set()
Self_risk_percep
Unexpected values:
set()
Missing expected categories:
set()


In [11]:
#Create the Encoder
ordinal_columns = list(ordinal_mapping.keys())
ordinal_categories = [
    ordinal_mapping[col]
    for col in ordinal_columns
]
ordinal_encoder = OrdinalEncoder(
    categories=ordinal_categories,
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=np.int64
)

#Fit ONLY on Training Data
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

X_train_encoded[ordinal_columns] = ordinal_encoder.fit_transform(
    X_train[ordinal_columns]
)

X_test_encoded[ordinal_columns] = ordinal_encoder.transform(
    X_test[ordinal_columns]
)

#Verify encoding
#print("Training object columns:")
#print(
#    X_train_encoded.select_dtypes(
#        include="object"
#    ).columns.tolist()
#)

#print("\nTesting object columns:")
#print(
#    X_test_encoded.select_dtypes(
#        include="object"
#    ).columns.tolist()
#)

In [12]:
#Encode gender
gender_mapping = {
    "Female": 0,
    "Male": 1
}
X_train_encoded["Gender"] = X_train_encoded["Gender"].map(gender_mapping)
X_test_encoded["Gender"] = X_test_encoded["Gender"].map(gender_mapping)

#Verify
print("Training Gender:")
print(X_train_encoded["Gender"].value_counts(dropna=False))

print("\nTesting Gender:")
print(X_test_encoded["Gender"].value_counts(dropna=False))

print("\nGender dtype:")
print(X_train_encoded["Gender"].dtype)


Training Gender:
Gender
0    241
1    214
Name: count, dtype: int64

Testing Gender:
Gender
0    66
1    48
Name: count, dtype: int64

Gender dtype:
int64


In [13]:
#Check for unexpected/missing values
print("Training Gender NaN:",
    X_train_encoded["Gender"].isna().sum())

print("Testing Gender NaN:",
    X_test_encoded["Gender"].isna().sum())

#verify columns
print("Training object columns:")
print(
    X_train_encoded.select_dtypes(
        include="object"
    ).columns.tolist()
)

print("\nTesting object columns:")
print(
    X_test_encoded.select_dtypes(
        include="object"
    ).columns.tolist()
)
#check NaN
print("Training NaN:", X_train_encoded.isna().sum().sum())
print("Testing NaN :", X_test_encoded.isna().sum().sum())
print("y_train NaN :", y_train.isna().sum())
print("y_test NaN  :", y_test.isna().sum())

#DataType check
print(X_train_encoded.dtypes.value_counts())

#check leakages
forbidden = [
    "CGPA",
    "GPA_S2",
    "GPA_LAST",
    "FAIL_COUNT",
    "WARN_STATUS",
    "REPEAT_COUNT"
]

found = [
    col for col in X_train_encoded.columns
    if col in forbidden
]

print("Forbidden variables found:")
print(found)

Training Gender NaN: 0
Testing Gender NaN: 0
Training object columns:
[]

Testing object columns:
[]
Training NaN: 0
Testing NaN : 0
y_train NaN : 0
y_test NaN  : 0
int64      52
float64     6
Name: count, dtype: int64
Forbidden variables found:
[]


In [14]:
#Save
X_train_encoded.to_csv(
    PROCESSED_DIR / "X_train_clean.csv",
    index=False
)

X_test_encoded.to_csv(
    PROCESSED_DIR / "X_test_clean.csv",
    index=False
)

y_train.to_csv(
    PROCESSED_DIR / "y_train_clean.csv",
    index=False
)

y_test.to_csv(
    PROCESSED_DIR / "y_test_clean.csv",
    index=False
)
#==============
train_clean = X_train_encoded.copy()
train_clean["RISK_LABEL"] = y_train.values

test_clean = X_test_encoded.copy()
test_clean["RISK_LABEL"] = y_test.values

train_clean.to_csv(
    PROCESSED_DIR / "train_clean.csv",
    index=False
)

test_clean.to_csv(
    PROCESSED_DIR / "test_clean.csv",
    index=False
)


print("Clean datasets successfully updated.")

Clean datasets successfully updated.
